# Fase 3 · M05: Target y Exportación

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 3 — Feature Engineering |
| **Módulo** | M05 — Target y Export |

---

## 🎯 Qué hace

Define el target `abandono` y exporta los datasets D y D_strict con los 4 casos de leakage auditados. Genera los datasets definitivos para modelado.

## 📋 Requisitos

- `data/03_features/df_expediente_features.parquet`
- `data/03_features/df_eda_con_target.parquet`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/automl/dataset_final_tfm.parquet` (D_strict) | Dataset definitivo para modelado |
| `data/automl/df_exp_automl_D.parquet` | Caso D (22 features) |
| `docs/html/fase3/m05_target_export.html` | Informe HTML |

## 🔄 Flujo

```
df_expediente_features.parquet
    ↓ Definición target abandono (>2 años sin actividad)
    ↓ Auditoría de leakage (casos A, B, D, D_strict)
    ↓ Exportación en parquet/csv/xlsx
    → dataset_final_tfm.parquet + df_exp_automl_D.parquet + HTML
```

## ➡️ Siguiente

`f3_m06_alerta_temprana.ipynb` — análisis de alerta temprana


Leakage (fuga de datos) es cuando el modelo tiene acceso a información que en la realidad no tendría en el momento de hacer la predicción. Es como hacer un examen con las respuestas delante.
En tu caso concreto: quieres predecir si un alumno va a abandonar la carrera. Pero en el dataset original hay columnas que ya contienen la respuesta:
egresado — Dice si el alumno se graduó (S/N). Si no se graduó y lleva años sin matricularse, es que abandonó. Esto ES el target disfrazado.
egresado_de_hecho — Igual, indica si terminó sin título oficial.
curso_ultimo — El último año en que se matriculó. Si 2021 - curso_ultimo >= 2, el alumno abandonó. Un árbol de decisión de profundidad 3 resolvía esto con AUC=1.0 — literalmente leía la fórmula del target.
anos_inactivo — Años sin matricularse. Es la fórmula del target directamente.
pct_titulacion y cred_faltantes — Porcentaje de carrera completado y créditos que le faltan. Implican saber si se graduó o no.
per_id_ficticio y exp_tit_id — Son identificadores, no aportan información predictiva.
Antes de eliminar estas columnas, todos los modelos AutoML daban métricas perfectas (AUC=1.0, F1=1.0). Parecía maravilloso pero era falso — el modelo no aprendía patrones de abandono, solo leía la respuesta. Después de eliminarlas, el mejor modelo baja a AUC=0.93, que es un resultado real y honesto.

# F3-M05: Target (Abandono) y Export Final

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## Qué hace

1. Calcular variable **TARGET (abandono)**
2. Eliminar columnas **leakage**
3. Exportar **3 casos** (A, B, C) a `data/automl/`

| Caso | Descripción | Qué quita |
|------|-------------|----------|
| **A** | Completo | Solo leakage directo |
| **B** | Sin casi_termino | + indicador_casi_termino |
| **C** | Sin temporales | + curso_inicio, duracion_real, n_cursos |

El **Caso C** simula un sistema de alerta temprana: predecir abandono
sin saber cuánto tiempo lleva matriculado el alumno.

## Requisitos
- `df_exp_automl_target.parquet` en `data/03_features/`
- `df_exp_target_eda.parquet` en `data/03_features/`

## Genera
- `data/automl/df_exp_automl_{A,B,C}.{parquet,csv,xlsx,json}`

## Flujo
M04a/M04b → **M05** → M06 (Alerta Temprana) → M07 (Validación) → M08 (Auditoría)

## Siguiente
- `f3_m06_alerta_temprana.ipynb`


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent


sys.path.insert(0, str(ROOT))

from src.config import RUTA_FEATURES, RUTA_AUTOML, RUTA_HTML, info_entorno
from src.utils import cargar_parquet, crear_directorios, formato_numero_es
from src.html import generar_kpis_html, generar_seccion_html, generar_html_navegacion_completa, guardar_html
from src.html.render import render_base_html

RUTA_FASE3_HTML = RUTA_HTML / 'fase3'
crear_directorios([RUTA_FEATURES, RUTA_AUTOML, RUTA_FASE3_HTML])

CURSO_REFERENCIA = 2021
ANOS_INACTIVO_UMBRAL = 2
fmt = formato_numero_es

print('✓ Configuración completada')
info_entorno()

✓ Directorios verificados: 3
✓ Configuración completada
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_pro

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATASETS
# ============================================================================

print('\n' + '='*70)
print('F3-M05: TARGET (ABANDONO) Y EXPORT FINAL')
print('='*70)

print('\n📥 Cargando datasets M04a (AutoML) y M04b (EDA)...')

df_automl = cargar_parquet(RUTA_FEATURES / 'df_exp_automl_target.parquet')
df_eda = cargar_parquet(RUTA_FEATURES / 'df_exp_target_eda.parquet')

print(f'✓ AutoML: {len(df_automl):,} × {len(df_automl.columns)} columnas')
print(f'✓ EDA: {len(df_eda):,} × {len(df_eda.columns)} columnas')


F3-M05: TARGET (ABANDONO) Y EXPORT FINAL

📥 Cargando datasets M04a (AutoML) y M04b (EDA)...
✓ Cargado: df_exp_automl_target.parquet (33,621 filas, 46 columnas)
✓ Cargado: df_exp_target_eda.parquet (33,621 filas, 47 columnas)
✓ AutoML: 33,621 × 46 columnas
✓ EDA: 33,621 × 47 columnas


In [3]:
# ============================================================================
# CELDA 3: CALCULAR TARGET (ABANDONO)
# ============================================================================

print('\n' + '-'*70)
print('CALCULANDO TARGET (ABANDONO)')
print('-'*70)

print(f'\nFórmula: abandono = (egresado="N") AND (egresado_de_hecho=0) AND ({CURSO_REFERENCIA} - curso_ultimo >= {ANOS_INACTIVO_UMBRAL})')

for df_name, df in [('AutoML', df_automl), ('EDA', df_eda)]:
    df['anos_inactivo'] = CURSO_REFERENCIA - df['curso_ultimo']
    egresado_val = df['egresado'].iloc[0]
    if isinstance(egresado_val, str):
        no_egresado = (df['egresado'] == 'N')
    else:
        no_egresado = (df['egresado'] == 0)
    no_egresado_de_hecho = True
    if 'egresado_de_hecho' in df.columns:
        no_egresado_de_hecho = (df['egresado_de_hecho'] == 0)
    abandono_condition = no_egresado & no_egresado_de_hecho & (df['anos_inactivo'] >= ANOS_INACTIVO_UMBRAL)
    df['abandono'] = abandono_condition.astype(int)

    n_total = len(df)
    n_abandono = (df['abandono'] == 1).sum()
    if isinstance(egresado_val, str):
        n_egresados = (df['egresado'] == 'S').sum()
    else:
        n_egresados = (df['egresado'] == 1).sum()
    n_de_hecho = (df['egresado_de_hecho'] == 1).sum() if 'egresado_de_hecho' in df.columns else 0
    n_activos = n_total - n_egresados - n_de_hecho - n_abandono

    print(f'\n✓ {df_name} ({fmt(n_total)} expedientes):')
    print(f'   🎓 Egresados (título):      {fmt(n_egresados)} ({n_egresados/n_total*100:.1f}%)')
    print(f'   📋 Completaron sin título:   {fmt(n_de_hecho)} ({n_de_hecho/n_total*100:.1f}%)')
    print(f'   ❌ Abandono (target=1):      {fmt(n_abandono)} ({n_abandono/n_total*100:.1f}%)')
    print(f'   ⏳ Activos (< 2 años):       {fmt(n_activos)} ({n_activos/n_total*100:.1f}%)')


----------------------------------------------------------------------
CALCULANDO TARGET (ABANDONO)
----------------------------------------------------------------------

Fórmula: abandono = (egresado="N") AND (egresado_de_hecho=0) AND (2021 - curso_ultimo >= 2)

✓ AutoML (33.621 expedientes):
   🎓 Egresados (título):      12.392 (36.9%)
   📋 Completaron sin título:   170 (0.5%)
   ❌ Abandono (target=1):      9.833 (29.2%)
   ⏳ Activos (< 2 años):       11.226 (33.4%)

✓ EDA (33.621 expedientes):
   🎓 Egresados (título):      12.392 (36.9%)
   📋 Completaron sin título:   170 (0.5%)
   ❌ Abandono (target=1):      9.833 (29.2%)
   ⏳ Activos (< 2 años):       11.226 (33.4%)


In [4]:
# ============================================================================
# CELDA 3b: ELIMINAR COLUMNAS LEAKAGE
# ============================================================================
# ╔═══════════════════════════════════════════════════════════════════╗
# ║  DATA LEAKAGE                                                    ║
# ║  Antes de eliminar: todos los modelos daban AUC=1.0000           ║
# ║  Un DecisionTree depth=3 resolvía:                               ║
# ║    Si curso_ultimo <= 2019 → abandono = 1                       ║
# ║  Esto es la fórmula del target disfrazada.                       ║
# ║                                                                   ║
# ║  Columnas eliminadas:                                             ║
# ║  1. egresado, egresado_de_hecho → en fórmula target              ║
# ║  2. curso_ultimo, anos_inactivo → IS the target                   ║
# ║  3. pct_titulacion, cred_faltantes → implican egresado            ║
# ║  4. per_id_ficticio, exp_tit_id → IDs, no features               ║
# ╚═══════════════════════════════════════════════════════════════════╝

print('\n' + '-'*70)
print('ELIMINANDO COLUMNAS LEAKAGE')
print('-'*70)

COLS_LEAKAGE = ['egresado', 'egresado_de_hecho', 'curso_ultimo',
                'anos_inactivo', 'pct_titulacion', 'cred_faltantes']
COLS_ID = ['per_id_ficticio', 'exp_tit_id']
COLS_DROP = COLS_LEAKAGE + COLS_ID

resumen_leakage = []
for df_name, df in [('AutoML', df_automl), ('EDA', df_eda)]:
    cols_antes = len(df.columns)
    cols_eliminar = [c for c in COLS_DROP if c in df.columns]
    df.drop(columns=cols_eliminar, inplace=True)
    cols_despues = len(df.columns)
    n_ab = (df['abandono'] == 1).sum()
    resumen_leakage.append({'dataset': df_name, 'antes': cols_antes, 'despues': cols_despues, 'eliminadas': len(cols_eliminar)})
    print(f'\n✓ {df_name}: {cols_antes} → {cols_despues} columnas (-{len(cols_eliminar)})')
    print(f'  Eliminadas: {cols_eliminar}')
    print(f'  Target abandono: {n_ab} ({n_ab/len(df)*100:.1f}%)')
    print(f'  ✅ Sin leakage' if not [c for c in COLS_LEAKAGE if c in df.columns] else f'  ⚠️ LEAKAGE RESTANTE')

print(f'\n✅ Leakage eliminado.')


----------------------------------------------------------------------
ELIMINANDO COLUMNAS LEAKAGE
----------------------------------------------------------------------

✓ AutoML: 48 → 40 columnas (-8)
  Eliminadas: ['egresado', 'egresado_de_hecho', 'curso_ultimo', 'anos_inactivo', 'pct_titulacion', 'cred_faltantes', 'per_id_ficticio', 'exp_tit_id']
  Target abandono: 9833 (29.2%)
  ✅ Sin leakage

✓ EDA: 49 → 41 columnas (-8)
  Eliminadas: ['egresado', 'egresado_de_hecho', 'curso_ultimo', 'anos_inactivo', 'pct_titulacion', 'cred_faltantes', 'per_id_ficticio', 'exp_tit_id']
  Target abandono: 9833 (29.2%)
  ✅ Sin leakage

✅ Leakage eliminado.


In [5]:
# ============================================================================
# CELDA 3c: HTML DOCUMENTACIÓN LEAKAGE
# ============================================================================

nav_fases, nav_modulos = generar_html_navegacion_completa(fase_activa='fase3', modulo_activo='m05')

r = resumen_leakage[0]
KPIS = [
    {'valor': fmt(len(df_automl)), 'titulo': 'Expedientes'},
    {'valor': f'{(df_automl["abandono"]==1).mean()*100:.1f}%', 'titulo': 'Abandono'},
    {'valor': str(r['eliminadas']), 'titulo': 'Cols Leakage'},
    {'valor': '3', 'titulo': 'Casos (A/B/C)'},
]

s_target = generar_seccion_html('🎯 Cálculo del Target', '''
<div style="background:#f7fafc;padding:20px;border-radius:10px;font-family:monospace;font-size:14px;">
  <strong>abandono</strong> = (egresado = "N") AND (egresado_de_hecho = 0) AND (2021 - curso_ultimo &gt;= 2)
</div>''')

s_leakage = generar_seccion_html('⚠️ Data Leakage', '''
<div style="background:#fff5f5;padding:15px;border-radius:10px;border-left:4px solid #e53e3e;margin-bottom:15px;">
  <strong>Antes:</strong> Todos los modelos AUC=1.0000. DecisionTree depth=3: Si curso_ultimo≤2019 → abandono=1</div>
<div style="background:#f0fff4;padding:15px;border-radius:10px;border-left:4px solid #38a169;">
  <strong>Después:</strong> Resultados reales en Fase AutoML.</div>''')

s_casos = generar_seccion_html('📦 Tres Casos de Estudio', '''
<table style="width:100%;border-collapse:collapse;">
<tr style="background:#3182ce;color:white;"><th style="padding:10px;">Caso</th><th>Descripción</th><th>Elimina</th><th>Cols AutoML</th><th>Objetivo</th></tr>
<tr><td style="padding:8px;font-weight:bold;">A</td><td>Completo</td><td>Solo leakage directo</td><td>39</td><td>Máximo poder predictivo</td></tr>
<tr style="background:#f7fafc;"><td style="padding:8px;font-weight:bold;">B</td><td>Sin casi_termino</td><td>+ indicador_casi_termino</td><td>38</td><td>¿Importa esta feature?</td></tr>
<tr><td style="padding:8px;font-weight:bold;">C</td><td>Sin temporales</td><td>+ curso_inicio, duracion_real, n_cursos</td><td>35</td><td>Alerta temprana (sin info temporal)</td></tr>
</table>
<p style="margin-top:15px;"><strong>Caso C</strong> simula un sistema de alerta temprana: ¿podemos predecir abandono <em>sin saber cuánto tiempo lleva matriculado</em>?</p>''')

html = render_base_html(titulo='M05: Target y Export', icono='🎯', subtitulo='Fase 3: Feature Engineering',
    nav_fases=nav_fases, nav_modulos=nav_modulos,
    contenido=generar_kpis_html(KPIS) + s_target + s_leakage + s_casos,
    notebook_nombre='f3_m05_target_export.ipynb', notebook_carpeta='fase3_features')
ruta_html = RUTA_FASE3_HTML / 'm05_target_export.html'
guardar_html(html, ruta_html)
print(f'\n✅ HTML: {ruta_html}')

✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\m05_target_export.html

✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\m05_target_export.html


In [6]:
# ============================================================================
# CELDA 4: EXPORTAR CASOS A, B, C A data/automl/
# ============================================================================
# Exporta los 3 casos SOLO a data/automl/ para uso en la fase AutoML.
# data/04_eda/ se gestiona exclusivamente desde f3_m08_auditoria.ipynb
# que genera df_eda_final con las columnas auditadas y texto legible.
# ============================================================================

print('\n' + '='*70)
print('EXPORTANDO CASOS A, B, C A data/automl/')
print('='*70)

CASOS = {
    'A': {'desc': 'Completo (sin leakage)', 'drop_extra': []},
    'B': {'desc': 'Sin indicador_casi_termino', 'drop_extra': ['indicador_casi_termino']},
    'C': {'desc': 'Sin temporales (alerta temprana)', 'drop_extra': ['indicador_casi_termino', 'curso_inicio', 'duracion_real', 'n_cursos']},
}

for caso, info in CASOS.items():
    print(f'\n--- CASO {caso}: {info["desc"]} ---')

    # Dataset AutoML (codificado)
    df_a = df_automl.copy()
    cols_extra = [c for c in info['drop_extra'] if c in df_a.columns]
    if cols_extra: df_a.drop(columns=cols_extra, inplace=True)

    for ext, func in [
        ('parquet', lambda d,r: d.to_parquet(r, index=False)),
        ('csv', lambda d,r: d.to_csv(r, sep=';', encoding='utf-8-sig', index=False, decimal=',')),
        ('xlsx', lambda d,r: d.to_excel(r, index=False, sheet_name='datos')),
        ('json', lambda d,r: d.to_json(r, orient='records', force_ascii=False)),
    ]:
        func(df_a, RUTA_AUTOML / f'df_exp_automl_{caso}.{ext}')

    print(f'  \U0001f4c2 automl/df_exp_automl_{caso}.* : {df_a.shape[0]:,} \u00d7 {df_a.shape[1]} cols')
    del df_a

# Compatibilidad: _final = copia de A
import shutil
shutil.copy2(RUTA_AUTOML / 'df_exp_automl_A.parquet', RUTA_AUTOML / 'df_exp_automl_target_final.parquet')
print('\n\u2713 Compatibilidad: _final = copia de A')

print(f'\n\u2705 Exportaci\u00f3n completada (solo a data/automl/)')



EXPORTANDO CASOS A, B, C A data/automl/

--- CASO A: Completo (sin leakage) ---


  📂 automl/df_exp_automl_A.* : 33,621 × 40 cols

--- CASO B: Sin indicador_casi_termino ---


  📂 automl/df_exp_automl_B.* : 33,621 × 39 cols

--- CASO C: Sin temporales (alerta temprana) ---


  📂 automl/df_exp_automl_C.* : 33,621 × 36 cols

✓ Compatibilidad: _final = copia de A

✅ Exportación completada (solo a data/automl/)


In [7]:
# ============================================================================
# CELDA 4b: EXPORTAR DATASET EDA CON TARGET A 03_features/
# ============================================================================
# df_eda tiene: target (celda 3) + leakage eliminado (celda 3b)
# Texto legible. Lo lee M08 para generar df_eda_final.
# TRAZABILIDAD: M04b > M05 (target+leakage) > 03_features/
# ============================================================================

ruta_eda_target = RUTA_FEATURES / "df_eda_con_target.parquet"
df_eda.to_parquet(ruta_eda_target, index=False)

print("\n" + "-"*70)
print("EXPORTAR DATASET EDA CON TARGET")
print("-"*70)
print(f"  Fichero:  {ruta_eda_target.name}")
print(f"  Destino:  data/03_features/")
print(f"  Forma:    {df_eda.shape[0]:,} x {df_eda.shape[1]} cols")
print(f"  Target:   abandono")
print(f"  Texto:    SI (categoricas legibles)")
print(f"  Consumidor: f3_m08_auditoria.ipynb")



----------------------------------------------------------------------
EXPORTAR DATASET EDA CON TARGET
----------------------------------------------------------------------
  Fichero:  df_eda_con_target.parquet
  Destino:  data/03_features/
  Forma:    33,621 x 41 cols
  Target:   abandono
  Texto:    SI (categoricas legibles)
  Consumidor: f3_m08_auditoria.ipynb


In [8]:
# ============================================================================
# CELDA 5: RESUMEN FINAL
# ============================================================================

print('\n' + '='*70)
print('\u2705 F3-M05 COMPLETADO')
print('='*70)
print(f'\U0001f4ca Registros: {len(df_automl):,}')

print(f'\n\U0001f4c1 ARCHIVOS GENERADOS:')
print(f'\n   \U0001f4c2 data/automl/')
for caso in ['A', 'B', 'C']:
    for ext in ['parquet', 'csv', 'xlsx', 'json']:
        print(f'   \u251c\u2500 df_exp_automl_{caso}.{ext}')

n_a = len(df_automl.columns)
print(f'\n\U0001f3af Casos de estudio:')
print(f'   A: {n_a} cols (completo)')
print(f'   B: {n_a-1} cols (sin indicador_casi_termino)')
print(f'   C: {n_a-4} cols (sin temporales: alerta temprana)')

print(f'\n\U0001f3af Siguiente: f3_m06_alerta_temprana.ipynb')
print('='*70)



✅ F3-M05 COMPLETADO
📊 Registros: 33,621

📁 ARCHIVOS GENERADOS:

   📂 data/automl/
   ├─ df_exp_automl_A.parquet
   ├─ df_exp_automl_A.csv
   ├─ df_exp_automl_A.xlsx
   ├─ df_exp_automl_A.json
   ├─ df_exp_automl_B.parquet
   ├─ df_exp_automl_B.csv
   ├─ df_exp_automl_B.xlsx
   ├─ df_exp_automl_B.json
   ├─ df_exp_automl_C.parquet
   ├─ df_exp_automl_C.csv
   ├─ df_exp_automl_C.xlsx
   ├─ df_exp_automl_C.json

🎯 Casos de estudio:
   A: 40 cols (completo)
   B: 39 cols (sin indicador_casi_termino)
   C: 36 cols (sin temporales: alerta temprana)

🎯 Siguiente: f3_m06_alerta_temprana.ipynb
